STEP 1 — LOAD PREPROCESSED DATASET

In [1]:
import pandas as pd
final_df = pd.read_csv(
    "preprocessed_resume_dataset.csv"
)

#### STEP 2 — CHECK DATASET

In [2]:
print(final_df.shape)
print(final_df.head())

(10035, 6)
                                         resume_text            category  \
0  EDUCATION\n© MBA - Executive Leadership © Bach...  Accountant resumes   
1  HOWARD GERRARD\n\ncoo ie me\n\nees |= Verifyin...  Accountant resumes   
2  SENIOR ACCOUNTANT\n= Qo ° in\ninfo@resumekrafi...  Accountant resumes   
3  Olivia Ogilvy, Accountant\n1515 Pacific Ave, L...  Accountant resumes   
4  Brooklyn, NY\nStephen Greet, CPA (128) 336-785...  Accountant resumes   

     subfolder     file_name  length  \
0  Bing_images  Image_10.png     937   
1  Bing_images  Image_11.jpg     357   
2  Bing_images  Image_12.jpg    2814   
3  Bing_images  Image_14.jpg     974   
4  Bing_images  Image_16.jpg    1521   

                                        cleaned_text  
0  education mba executive leadership bachelor sc...  
1  howard gerrard coo ie ee verifying financial d...  
2  senior accountant qo info resumekraficom chica...  
3  olivia ogilvy accountant pacific ave los angel...  
4  brooklyn ny s

#### STEP 3 - Check columns

In [3]:
print(final_df.columns)

Index(['resume_text', 'category', 'subfolder', 'file_name', 'length',
       'cleaned_text'],
      dtype='object')


#### STEP 4 — CHECK CATEGORY COLUMN
##### This is the target column

In [4]:
print(
    final_df['category'].unique()
)

['Accountant resumes' 'Advocate resumes' 'Agricultural resumes'
 'Apparel resumes' 'Architects resumes' 'Arts resumes'
 'Automobile resumes' 'Aviation resumes' 'Banking resumes'
 'Blockchain resumes' 'BPO resumes' 'Building _Construction resumes'
 'Business Analyst resumes' 'Civil Engineer resumes' 'Consultant resumes'
 'data science resumes' 'Database resumes' 'Designing resumes'
 'DevOps Engineer resumes' 'Digital Media resumes'
 'DotNet Developer resumes' 'Education resumes'
 'Electrical Engineering resumes' 'ETL Developer resumes'
 'Finance resumes' 'Food_Beverages resumes' 'Health_Fitness resumes'
 'HR resumes' 'Information Technology resumes' 'Java Developer resumes'
 'Managment resumes' 'Mechanical Engineer resumes'
 'Network Security Engineer resumes' 'Operations Manager resumes'
 'PMO resumes' 'Public Relations resumes' 'Python Developer resumes'
 'React Developer resumes' 'Sales resumes' 'SAP Developer resumes'
 'SQL Developer resumes' 'Testing resumes' 'web designing resumes

#### STEP 5 — LABEL ENCODING
##### Deep learning models cannot understand text labels.Convert categories into numbers.

In [5]:
from sklearn.preprocessing import LabelEncoder
encoder = LabelEncoder()
final_df['label'] = encoder.fit_transform(
    final_df['category']
)

#### STEP 6 — VERIFY LABELS

In [6]:
print(
    final_df[
        ['category', 'label']
    ].head(10)
)

             category  label
0  Accountant resumes      1
1  Accountant resumes      1
2  Accountant resumes      1
3  Accountant resumes      1
4  Accountant resumes      1
5  Accountant resumes      1
6  Accountant resumes      1
7  Accountant resumes      1
8  Accountant resumes      1
9  Accountant resumes      1


#### STEP 7 — CHECK NUMBER OF CLASSES

In [7]:
num_classes = final_df[
    'label'
].nunique()

print(f"{num_classes} resume categories")

99 resume categories


#### STEP 8 — DEFINE FEATURES & TARGET

##### X = Input Text
##### y = Labels

In [8]:
X = final_df['cleaned_text']
y = final_df['label']

#### STEP 9 — TRAIN TEST SPLIT

In [9]:
from sklearn.model_selection import train_test_split
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42
)

#### STEP 10 — VERIFY SPLIT

In [10]:
print(X_train.shape)

print(X_test.shape)

print(y_train.shape)

print(y_test.shape)

(8028,)
(2007,)
(8028,)
(2007,)


#### STEP 11 — Install transformers

In [11]:
pip install transformers datasets torch

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.3 -> 26.1.2
[notice] To update, run: C:\Users\vanda\PycharmProjects\Assignment_1\env\scripts\python.exe -m pip install --upgrade pip


#### STEP 12 — Import BERT libraries

In [12]:
from transformers import (
    DistilBertTokenizerFast,
    DistilBertForSequenceClassification,
    Trainer,
    TrainingArguments
)

import torch

#### STEP 13 — Load tokenizer

In [13]:
tokenizer = DistilBertTokenizerFast.from_pretrained(
    "distilbert-base-uncased"
)

#### STEP 14 — Tokenize resumes

In [14]:
train_encodings = tokenizer(
    list(X_train),
    truncation=True,
    padding=True,
    max_length=256
)

test_encodings = tokenizer(
    list(X_test),
    truncation=True,
    padding=True,
    max_length=256
)

#### STEP 15 — Create PyTorch dataset class

In [15]:
class ResumeDataset(torch.utils.data.Dataset):

    def __init__(self, encodings, labels):

        self.encodings = encodings
        self.labels = labels

    def __getitem__(self, idx):

        item = {
            key: torch.tensor(val[idx])
            for key, val in self.encodings.items()
        }

        item["labels"] = torch.tensor(
            self.labels.iloc[idx]
        )

        return item

    def __len__(self):

        return len(self.labels)

#### STEP 16 — Create train/test datasets

In [16]:
train_dataset = ResumeDataset(
    train_encodings,
    y_train
)

test_dataset = ResumeDataset(
    test_encodings,
    y_test
)

#### STEP 17 — Load DistilBERT model

In [17]:
model = DistilBertForSequenceClassification.from_pretrained(
    "distilbert-base-uncased",
    num_labels=num_classes
)

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

[transformers] DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_layer_norm.bias   | UNEXPECTED | 
vocab_layer_norm.weight | UNEXPECTED | 
vocab_projector.bias    | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
vocab_transform.weight  | UNEXPECTED | 
pre_classifier.bias     | MISSING    | 
classifier.bias         | MISSING    | 
classifier.weight       | MISSING    | 
pre_classifier.weight   | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


#### STEP 18 — Training configuration

In [18]:
pip install accelerate>=1.1.0

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.3 -> 26.1.2
[notice] To update, run: C:\Users\vanda\PycharmProjects\Assignment_1\env\scripts\python.exe -m pip install --upgrade pip


In [19]:
training_args = TrainingArguments(
    output_dir="./results",

    num_train_epochs=4,

    per_device_train_batch_size=8,

    per_device_eval_batch_size=8,

    eval_strategy="epoch",

    save_strategy="epoch",

    load_best_model_at_end=True
)

#### STEP 19 — Create trainer

In [20]:
trainer = Trainer(
    model=model,

    args=training_args,

    train_dataset=train_dataset,

    eval_dataset=test_dataset
)

#### STEP 20 — Train model

In [21]:
trainer.train()

C:\Users\vanda\PycharmProjects\Assignment_1\env\Lib\site-packages\torch\utils\data\dataloader.py:752: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


Epoch,Training Loss,Validation Loss
1,1.687696,1.389510
2,1.016138,1.091186
3,0.762534,1.023402
4,0.607455,0.982627


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

C:\Users\vanda\PycharmProjects\Assignment_1\env\Lib\site-packages\torch\utils\data\dataloader.py:752: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

C:\Users\vanda\PycharmProjects\Assignment_1\env\Lib\site-packages\torch\utils\data\dataloader.py:752: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

C:\Users\vanda\PycharmProjects\Assignment_1\env\Lib\site-packages\torch\utils\data\dataloader.py:752: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

TrainOutput(global_step=4016, training_loss=1.2741161796201272, metrics={'train_runtime': 43370.827, 'train_samples_per_second': 0.74, 'train_steps_per_second': 0.093, 'total_flos': 2130575780339712.0, 'train_loss': 1.2741161796201272, 'epoch': 4.0})

#### STEP 21 — Evaluate model

In [22]:
results = trainer.evaluate()

print(results)

C:\Users\vanda\PycharmProjects\Assignment_1\env\Lib\site-packages\torch\utils\data\dataloader.py:752: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


Training Loss,Validation Loss,Epoch
0.607455,0.982627,4


{'eval_loss': 0.9826273322105408}


#### STEP 22 — Predict a resume

In [24]:
text = "Python developer with Django and AWS experience"

inputs = tokenizer(
    text,
    return_tensors="pt",
    truncation=True,
    padding=True
)

outputs = model(**inputs)

predicted_class = outputs.logits.argmax().item()

print(
    encoder.inverse_transform([predicted_class])
)

['PythonDeveloper']


#### STEP 23 — Save model

In [25]:
model.save_pretrained("./resume_classifier")

tokenizer.save_pretrained("./resume_classifier")

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

('./resume_classifier\\tokenizer_config.json',
 './resume_classifier\\tokenizer.json')

In [27]:
import os

print(os.listdir("resume_classifier"))

['config.json', 'model.safetensors', 'tokenizer.json', 'tokenizer_config.json']


In [28]:
import joblib

joblib.dump(encoder, "label_encoder.pkl")

['label_encoder.pkl']

In [29]:
import joblib

loaded_encoder = joblib.load("label_encoder.pkl")
print(len(loaded_encoder.classes_))

99


In [33]:
import joblib

joblib.dump(
    encoder,
    "label_encoder.pkl"
)

print("Encoder Saved")

Encoder Saved


In [34]:
import joblib

encoder = joblib.load("label_encoder.pkl")

print(len(encoder.classes_))

99
